In [1]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pyspark.sql import functions as F
from spark_session import get_spark
from config import GCS_BUCKET

spark = get_spark("Gold Processing")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 16:09:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
SILVER_PATH = f"gs://{GCS_BUCKET}/silver"
GOLD_PATH   = f"gs://{GCS_BUCKET}/gold"

silver_df = spark.read.format("delta").load(SILVER_PATH)
silver_df.show(5)

26/05/15 16:10:04 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/05/15 16:10:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+--------------------+----------------+-------+---------+-----------+-----+--------------+----------+----+-------+----+--------+----------------+-----------+---------+-------------+--------+----------+-----------+----------+
|          id|           road_name|  recordDatetime|density|occupancy|waitingTime|speed|sampledSeconds|traveltime|flow|entered|left|timeloss| processDatetime|temperature|windspeed|precipitation| weather|      date|laneDensity|event_time|
+------------+--------------------+----------------+-------+---------+-----------+-----+--------------+----------+----+-------+----+--------+----------------+-----------+---------+-------------+--------+----------+-----------+----------+
|-600018755#4|Ngõ 130 Đường Xuâ...|2026-05-04T00:06|   0.41|      0.2|        0.0|13.56|          0.47|       0.1|20.0|    1.0| 1.0|    0.01|2026-05-15T14:13|       17.0|      7.2|          0.0|Overcast|2026-05-04|       NULL|      NULL|
|-600018755#0|Ngõ 130 Đường Xuâ...|2026-05-04T00

In [11]:
# Hiển thị 20 dòng đầu tiên của kết quả
silver_df.groupBy("road_name", "recordDatetime").count().show()

+--------------------+----------------+-----+
|           road_name|  recordDatetime|count|
+--------------------+----------------+-----+
|Ngõ 2 Phố Đặng Th...|2026-05-15T21:57|    5|
|Ngõ 45 Trần Thái ...|2026-05-15T21:57|    6|
|    Phố Khúc Thừa Dụ|2026-05-15T21:57|   29|
|   Ngõ 4 Phố Duy Tân|2026-05-15T21:57|    1|
|   Phố Doãn Kế Thiện|2026-05-15T21:57|    6|
|   Phố Dịch Vọng Hậu|2026-05-15T21:57|   17|
|         Phố Tô Hiệu|2026-05-15T21:57|   22|
|Đường Nguyễn Văn ...|2026-05-15T21:57|   50|
|  Ngõ 15 Phố Duy Tân|2026-05-15T21:57|    4|
|Ngõ 10 Đường Nguy...|2026-05-15T21:57|    8|
|    395 Nguyễn Khang|2026-05-15T21:57|    5|
|Ngõ 143 Phố Quan Hoa|2026-05-15T21:57|    4|
|Ngõ 25 Phố Dịch V...|2026-05-15T21:57|    1|
|Ngõ 49 Phố Trần Đ...|2026-05-15T21:57|    1|
|         Phố Tô Hiệu|2026-05-15T22:45|   22|
|Đường Nguyễn Phon...|2026-05-15T22:45|   51|
|       Phố Đông Quan|2026-05-15T22:45|   12|
|      Ngõ 15 Duy Tân|2026-05-15T22:45|    4|
|         Phố Chùa Hà|2026-05-15T2

In [14]:
gold= spark.read.format("delta").load(GOLD_PATH)

In [15]:
gold.show(5)

+--------------------+----------------+---------+-----------+-------------+---------------+--------------+--------+-------------+----------+------------+-----------+---------+-------------+------------+----------+
|           road_name|  recordDatetime|avg_speed|avg_density|avg_occupancy|avg_waitingTime|avg_traveltime|avg_flow|total_entered|total_left|avg_timeloss|temperature|windspeed|precipitation|     weather|      date|
+--------------------+----------------+---------+-----------+-------------+---------------+--------------+--------+-------------+----------+------------+-----------+---------+-------------+------------+----------+
|   Cầu vượt Mai Dịch|2026-05-15T21:57|     2.16|       NULL|         0.18|           19.0|          9.87|   19.49|          1.0|       1.0|        NULL|       29.9|     13.2|         NULL|Thunderstorm|2026-05-15|
|Ngõ 1 Phố Dịch Vọ...|2026-05-15T21:57|    11.23|       NULL|         0.67|            0.0|          3.46|   66.67|         10.0|      10.0|    

In [5]:
gold_df = silver_df.groupBy("road_name", "recordDatetime").agg(
    F.round(F.avg("speed"),       2).alias("avg_speed"),
    F.round(F.avg("density"),     2).alias("avg_density"),
    F.round(F.avg("occupancy"),   2).alias("avg_occupancy"),
    F.round(F.avg("waitingTime"), 2).alias("avg_waitingTime"),
    F.round(F.avg("traveltime"),  2).alias("avg_traveltime"),
    F.round(F.avg("flow"),        2).alias("avg_flow"),
    F.round(F.sum("entered"),     2).alias("total_entered"),
    F.round(F.sum("left"),        2).alias("total_left"),
    F.round(F.avg("timeloss"),    2).alias("avg_timeloss"),
    F.first("temperature").alias("temperature"),
    F.first("windspeed").alias("windspeed"),
    F.first("precipitation").alias("precipitation"),
    F.first("weather").alias("weather"),
)

gold_df.show(5)

+--------------------+----------------+---------+-----------+-------------+---------------+--------------+--------+-------------+----------+------------+-----------+---------+-------------+--------+
|           road_name|  recordDatetime|avg_speed|avg_density|avg_occupancy|avg_waitingTime|avg_traveltime|avg_flow|total_entered|total_left|avg_timeloss|temperature|windspeed|precipitation| weather|
+--------------------+----------------+---------+-----------+-------------+---------------+--------------+--------+-------------+----------+------------+-----------+---------+-------------+--------+
|Ngách 2 Ngõ 10 Ng...|2026-05-04T00:06|    11.55|       0.48|          0.1|            0.0|         10.14|    20.0|          1.0|       0.0|        0.39|       17.0|      7.2|          0.0|Overcast|
|Ngõ 130 Đường Xuâ...|2026-05-04T00:06|    12.99|       0.38|         0.21|            0.0|          1.23|   17.87|          5.0|       6.0|        0.26|       17.0|      7.2|          0.0|Overcast|
|    

In [7]:
gold_df.count()

33

In [6]:
(
    gold_df
    .withColumn("date", F.to_date("recordDatetime", "yyyy-MM-dd'T'HH:mm"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("date")
    .save(GOLD_PATH)
)

print("Saved to Gold.")

Saved to Gold.
